In [ ]:
# Imports e configuração
import os
import re
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from transformers import pipeline


In [ ]:
# Carregar dados
csv_in_path = 'voc_silver_2025.csv'
df = pd.read_csv(csv_in_path)

df['data'] = pd.to_datetime(df['data'], errors='coerce')
df['origem'] = df['origem'].astype(str).str.strip()
df['usuario'] = df['usuario'].astype(str).str.strip()
df['mensagem'] = df['mensagem'].astype(str).fillna('')

print(df.head())
print(df.shape)


In [ ]:
# Configurar modelo Hugging Face com fallback
use_hf = True
hf_error = None
sentiment_pipe = None

try:
    sentiment_pipe = pipeline(
        task='sentiment-analysis',
        model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
        top_k=None
    )
except Exception as err:
    use_hf = False
    hf_error = str(err)

print(use_hf)
if hf_error is not None:
    print(hf_error)


In [ ]:
# Funções auxiliares
POS_WORDS = ['bom','boa','otimo','ótimo','excelente','perfeito','obrigado','obrigada','show','resolvido','funcionou','sucesso','rapido','rápido']
NEG_WORDS = ['erro','bug','falha','caiu','fora do ar','instavel','instável','lento','lenta','demora','sem retorno','sem resposta','nao funciona','não funciona','problema','travando','trava','pessimo','péssimo','horrivel','horrível','inaceitavel','inaceitável','urgente','401','500','timeout','rejeitado','recusado']

def rule_sentiment(text_val):
    t = str(text_val).lower()
    pos_hits = sum([1 for w in POS_WORDS if w in t])
    neg_hits = sum([1 for w in NEG_WORDS if w in t])
    if neg_hits > pos_hits and neg_hits > 0:
        return 'NEGATIVA'
    if pos_hits > neg_hits and pos_hits > 0:
        return 'POSITIVA'
    return 'NEUTRA'

CAT_RULES = [
    ('INSTABILIDADE NA PLATAFORMA', ['fora do ar','instavel','instável','caiu','travando','trava','lento','lenta','timeout','latencia','latência','500','502','503']),
    ('API/AUTENTICACAO', ['api','token','401','403','autenticacao','autenticação','oauth','jwt']),
    ('BUGS', ['bug','erro','falha','stack','excecao','exceção','quebrou']),
    ('SUPORTE/ATENDIMENTO', ['sem retorno','sem resposta','demora','atraso','chamado','ticket','suporte','atendimento']),
    ('FINANCEIRO', ['fatura','cobranca','cobrança','boleto','pix','nota fiscal','nf','reembolso','pagamento','preco','preço','plano']),
    ('TREINAMENTO/ONBOARDING', ['treinamento','trilha','onboarding','curso','como usar','tutorial','documentacao','documentação']),
    ('FEATURE REQUEST', ['seria bom','poderia ter','faltou','queria','gostaria','sugestao','sugestão','melhoria','feature'])
]

def categorize(text_val):
    t = str(text_val).lower()
    for cat, kws in CAT_RULES:
        for kw in kws:
            if kw in t:
                return cat
    return 'FEEDBACK GERAL'

def hf_to_pt(label_val):
    if label_val == 'LABEL_0':
        return 'NEGATIVA'
    if label_val == 'LABEL_2':
        return 'POSITIVA'
    return 'NEUTRA'


In [ ]:
# Classificação de sentimento
texts = df['mensagem'].astype(str).tolist()
sent_out = []

if sentiment_pipe is not None:
    batch_size = 64
    for start_idx in tqdm(range(0, len(texts), batch_size)):
        batch_txt = texts[start_idx:start_idx+batch_size]
        preds = sentiment_pipe(batch_txt)
        for item in preds:
            best = sorted(item, key=lambda x: x['score'], reverse=True)[0]
            sent_out.append(hf_to_pt(best['label']))
else:
    for t in tqdm(texts):
        sent_out.append(rule_sentiment(t))

df['sentimentalidade'] = sent_out


In [ ]:
# Categorizar
df['categoria'] = [categorize(x) for x in tqdm(df['mensagem'].astype(str).tolist())]


In [ ]:
# Exportar resultado
csv_out_path = 'voc_gold_2025.csv'
df.to_csv(csv_out_path, index=False)
print(csv_out_path)
